# OCEAN Personality Model — LoRA Fine-Tuning

**Base model:** `Qwen/Qwen2.5-1.5B-Instruct` (~3 GB in bf16, well under 4 GB limit)  
**Method:** LoRA (no quantization, no QLoRA)  
**Dataset:** `Datasets/Ocean.json`  
**Goal:** Fine-tune the model to generate structured OCEAN personality reports so it can replace the Gemini API call in `OceanModel.py`

---
### VRAM requirements
| Setup | VRAM needed |
|---|---|
| Full LoRA training (bf16) | ~10–12 GB |
| With gradient checkpointing | ~7–8 GB |
| Inference only (after training) | ~3.5 GB |

If you're on Colab, use a T4/A100 GPU runtime. On a Mac with Apple Silicon, MPS backend will be used automatically.

## 1. Install Dependencies

In [ ]:
# Run once — restart kernel after installing
%pip install -q transformers==4.47.0 peft==0.14.0 trl==0.13.0 accelerate==1.2.0 datasets bitsandbytes torch

## 2. Imports

In [ ]:

import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig

# Detect device
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float16   # MPS doesn't support bf16 well
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

print(f"Using device: {DEVICE} | dtype: {DTYPE}")

## 3. Load & Prepare Dataset

In [ ]:
DATASET_PATH = "Datasets/Ocean.json"

with open(DATASET_PATH, "r") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} examples")
print("Sample entry:")
print(json.dumps(raw_data[0], indent=2))

In [ ]:
# Format each entry into a chat-style prompt.
# Qwen2.5 uses the ChatML format natively.

SYSTEM_PROMPT = (
    "You are an objective psychometrician and career counselor. "
    "You analyze Big Five (OCEAN) personality scores (0-100 normalized scale, "
    "50 = average) and produce structured, honest, plain-language reports."
)

def format_example(entry):
    """Convert instruction/input/output entry to ChatML format."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"{entry['instruction']}\n\n{entry['input']}"},
        {"role": "assistant", "content": entry["output"]},
    ]
    return {"messages": messages}

formatted = [format_example(e) for e in raw_data]
dataset = Dataset.from_list(formatted)

# 90/10 train/validation split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

## 4. Load Base Model & Tokenizer

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # required for SFTTrainer

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True,
)

if DEVICE != "cuda":
    model = model.to(DEVICE)

# Enable gradient checkpointing to save VRAM during training
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

print(f"Model loaded. Parameters: {model.num_parameters() / 1e9:.2f}B")

## 5. Apply LoRA

We target the attention projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and the feed-forward layers (`gate_proj`, `up_proj`, `down_proj`). This is standard practice for instruction fine-tuning.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                        # rank — higher = more capacity, more VRAM
    lora_alpha=32,               # scaling factor (alpha/r = 2 is typical)
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~1-2% of total params — only the LoRA adapters are trained

## 6. Training Configuration

In [ ]:
OUTPUT_DIR = "./ocean_lora_adapter"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 2*4 = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=(DTYPE == torch.float16),
    bf16=(DTYPE == torch.bfloat16),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    max_seq_length=512,
    dataset_text_field=None,          # we pass messages directly
    packing=False,
)

## 7. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

## 8. Save the LoRA Adapter

The adapter is a small set of weight matrices (~50-200 MB), NOT the full model.

In [ ]:
ADAPTER_SAVE_PATH = "./ocean_lora_adapter/final"

trainer.model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"LoRA adapter saved to: {ADAPTER_SAVE_PATH}")

# Check adapter size
import os
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, f))
    for f in os.listdir(ADAPTER_SAVE_PATH) if os.path.isfile(os.path.join(ADAPTER_SAVE_PATH, f))
)
print(f"Adapter size on disk: {total_size / 1e6:.1f} MB")

## 9. (Optional) Merge Adapter into Base Model & Save Full Model

Merging creates a single standalone model file (~3 GB).  
Use this if you want a self-contained model with no dependency on the base model at runtime.

In [ ]:
MERGED_SAVE_PATH = "./ocean_model_merged"

# Load base model fresh (unquantized) for merging
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="cpu",          # merge on CPU to avoid VRAM pressure
    trust_remote_code=True,
)

# Load adapter and merge weights
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_PATH)
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(MERGED_SAVE_PATH)
tokenizer.save_pretrained(MERGED_SAVE_PATH)

print(f"Merged model saved to: {MERGED_SAVE_PATH}")

## 10. Test Inference

In [ ]:
# Load either the adapter path or the merged model path
INFERENCE_PATH = ADAPTER_SAVE_PATH   # swap to MERGED_SAVE_PATH if you ran step 9

inf_tokenizer = AutoTokenizer.from_pretrained(INFERENCE_PATH, trust_remote_code=True)

inf_model = AutoModelForCausalLM.from_pretrained(
    INFERENCE_PATH,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True,
)
if DEVICE != "cuda":
    inf_model = inf_model.to(DEVICE)
inf_model.eval()

# --- Run a test ---
test_scores = {
    "Openness": 72,
    "Conscientiousness": 55,
    "Extraversion": 30,
    "Agreeableness": 68,
    "Neuroticism": 45,
}

scores_text = "\n".join([f"- {k}: {v}" for k, v in test_scores.items()])
user_message = (
    "Generate a professional personality report based on the provided OCEAN scores.\n\n"
    f"{scores_text}"
)

messages = [
    {"role": "system",  "content": SYSTEM_PROMPT},
    {"role": "user",    "content": user_message},
]

# Apply Qwen2.5 chat template
prompt = inf_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = inf_tokenizer(prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    output_ids = inf_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=inf_tokenizer.eos_token_id,
    )

# Decode only the generated tokens (not the prompt)
generated = inf_tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("=" * 60)
print("MODEL OUTPUT:")
print("=" * 60)
print(generated)

## 11. How to Integrate with `OceanModel.py`

Replace the `llm_analysis` function in `OceanModel.py` with the following local model version:

```python
# OceanModel.py — local model version (no API key needed)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "./ocean_lora_adapter/final"   # or "./ocean_model_merged"

SYSTEM_PROMPT = (
    "You are an objective psychometrician and career counselor. "
    "You analyze Big Five (OCEAN) personality scores (0-100 normalized scale, "
    "50 = average) and produce structured, honest, plain-language reports."
)

_tokenizer = None
_model = None

def _load_model():
    global _tokenizer, _model
    if _model is None:
        device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
        dtype = torch.bfloat16 if device == "cuda" else torch.float16 if device == "mps" else torch.float32
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
        _model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH, torch_dtype=dtype, device_map="auto" if device == "cuda" else None, trust_remote_code=True
        )
        if device != "cuda":
            _model = _model.to(device)
        _model.eval()
    return _tokenizer, _model

def llm_analysis(scores: dict) -> str:
    tokenizer, model = _load_model()
    device = next(model.parameters()).device
    scores_text = "\n".join([f"- {k}: {v}" for k, v in scores.items()])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Generate a professional personality report based on the provided OCEAN scores.\n\n{scores_text}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, temperature=0.7, top_p=0.9,
            do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
```

The model is lazy-loaded on first call and cached for subsequent calls — so Streamlit's rerun won't reload it every time.